In [ ]:
import rl
import torch
from tqdm import tqdm
from rl import RLVR, TrainingConfig
from datasets import load_from_disk
from rlvr_dataset import RLVRDataset
from rl.rewards import BinaryRewards
from model.model_interface import ModelInterface

## Constants

In [ ]:
# dataset config
dataset_path = "/Users/towrist/root/rlvr/data/gsm8K"
dataset_seed = 0
validation_fraction = 0.2
R1_ZERO_PROMPT = """A conversation between User and Assistant. The User asks a question, and the Assistant solves it.\
 The Assistant first thinks about the reasoning process in the mind and then provides the\
 User with the answer. The reasoning process is enclosed within <think> </think> and the\
 answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning\
 process here </think> <answer> answer here </answer>.\
 User: {question}\
 Assistant: <think>"""

# model loading config
model_path = "/Users/towrist/root/rlvr/models/OLMo-2-0425-1B"
device = "mps"
model_dtype = torch.bfloat16

# training_config
sampling_temperature = 1
sampling_group_size = 2
max_rollout_tokens = 526
stop_strings = ["</answer>"]

n_iters = 500

## Dataset Prep

### Load GSM8K

In [ ]:
gsm8K = load_from_disk(dataset_path)

### Convert to RLVRDataset

In [ ]:
def gsm8K_rlvr_answer_extractor(answers: str) -> list[float]:
    if isinstance(answers, str):
        answers = [answers]
    return [float(answer.rsplit("####", 1)[1].replace(",", ""))\
                for answer in answers]

### r1-zero prompt injector

In [ ]:
def r1_zero_prompt_injector(prompts: list[str]) -> list[str]:
    return [R1_ZERO_PROMPT.replace("{question}", prompt) for prompt in prompts]

In [ ]:
train_indices, valid_indices = RLVRDataset.perform_train_valid_split(gsm8K['train'].num_rows, 
                                                                     validation_fraction = validation_fraction, 
                                                                     split_seed = dataset_seed)

dtrain = RLVRDataset(
                         seed = dataset_seed,
                         prompts = list(r1_zero_prompt_injector(gsm8K['train']['question'][train_indices])),
                         answers = list(gsm8K_rlvr_answer_extractor(gsm8K['train']['answer'][train_indices]))
                    )
dvalid = RLVRDataset(
                         seed = dataset_seed,
                         prompts = list(r1_zero_prompt_injector(gsm8K['train']['question'][valid_indices])),
                         answers = list(gsm8K_rlvr_answer_extractor(gsm8K['train']['answer'][valid_indices]))
                     )
dtest = RLVRDataset(
                        seed = dataset_seed,
                        prompts = list(r1_zero_prompt_injector(gsm8K['test']['question'])),
                        answers = list(gsm8K_rlvr_answer_extractor(gsm8K['test']['answer']))
                    )

## Training Config Generation

### Reward Function

In [ ]:
def rollout_answer_extractor(rollout_prompt: str) -> float:
    if (not "<answer>" in rollout_prompt) or (not "</answer>" in rollout_prompt):
        return float('nan')
    answer_str = rollout_prompt.split("<answer>", 1)[1].rsplit("</answer>", 1)[0].replace(",", "").strip()
    try:
        answer_float = float(answer_str)
    except:
        answer_float = float('nan')
    return answer_float
reward_function = BinaryRewards(rollout_answer_extractor)

In [ ]:
training_config = TrainingConfig(
                                    sampling_temperature = sampling_temperature,
                                    sampling_group_size = sampling_group_size,
                                    max_rollout_tokens = max_rollout_tokens,
                                    reward_function = reward_function,
                                    stop_strings = stop_strings,
                                )

## Model Interface

In [ ]:
model_interface = ModelInterface(
                                    local_model_path = model_path,
                                    device = "mps",
                                    model_dtype = torch.bfloat16
                                )

## RLVR object

In [ ]:
rlvr = RLVR(training_config, model_interface)

# Training

In [ ]:
%%time
for _ in tqdm(range(n_iters)):
    rlvr.train_one_step(**dtrain.get_batch(5))